# Sentiment Classification Project

In [1]:
import ctypes
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libcudart.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvrtc.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvJitLink.so", mode=ctypes.RTLD_GLOBAL)

import sys
sys.path.insert(0, "/home/taekim/.local/lib/python3.14/site-packages")

import torch
print(torch.__file__)
print(torch.version.cuda)
x = torch.randn(3, 3).cuda()
print(x @ x)

/home/taekim/.local/lib/python3.14/site-packages/torch/__init__.py
12.8
tensor([[-0.4189,  0.9998,  0.9181],
        [ 2.3182, -3.8194, -0.0356],
        [ 1.6785,  1.1948, -4.0689]], device='cuda:0')


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
import torch
x = torch.randn(10, 10).cuda()
print(x @ x)

tensor([[-1.0300,  0.0963, -1.0513, -4.4934,  4.8616, -0.1058,  1.4049,  0.0718,
         -4.2381, -3.7317],
        [-0.7236, -1.9065, -0.4759,  0.4046, -1.0793, -2.5741, -2.0516, -1.3444,
          1.0442,  1.2772],
        [ 3.2906,  1.5974, -1.0463,  4.8686,  3.3582,  3.5519,  0.1390,  2.5392,
         -0.2274,  4.5230],
        [-3.0495,  3.0791, -3.4386,  1.6366,  2.0315,  2.7518,  3.3286,  6.6044,
          3.4313,  7.8367],
        [-0.7530,  5.0194,  2.4396, -2.3647,  6.6984,  5.3907,  5.8480,  0.9684,
         -2.5129, -0.6345],
        [ 3.6895, -0.7803, -0.8411,  2.3508,  3.2283, -1.7174, -3.3668, -1.6578,
         -2.6283, -4.2989],
        [ 3.4548,  2.6941, -2.7923,  5.4302,  3.0706,  2.6632, -4.9426,  1.0481,
         -1.9053,  0.8672],
        [ 0.4708, -4.4724,  2.9211, -2.8046, -3.4494, -2.0574, -0.4936, -5.6814,
          0.1435,  1.2379],
        [-8.2449,  5.6555,  0.2314,  0.7254,  2.5338,  3.1849,  0.2174,  7.1552,
         -2.2208,  1.2512],
        [ 2.1845,  

# Verify Setup
Make sure cuda (GPU) is available

In [4]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Ti


# Load data

In [5]:
train_full = pd.read_csv("/cluster/courses/cil/text-classification/data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [6]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [7]:
# # so that it reloads the modules every time we run this cell, so we always have the latest version of the code
# %load_ext autoreload
# %autoreload 2

# import importlib
# import train_loop
# importlib.reload(train_loop)
# from train_loop import train_loop

In [8]:
# NOTE: don't do GloVe, as only for English
# downlaod GloVe embeddings (100-dimensional vectors) and extract the file
# import urllib.request
# import zipfile

# url = "http://nlp.stanford.edu/data/glove.6B.zip"
# urllib.request.urlretrieve(url, "glove.6B.zip")
# with zipfile.ZipFile("glove.6B.zip", "r") as f:
#     f.extract("glove.6B.100d.txt")  # 100-dimensional vectors

## Simple baselines

As baselines, we first evaluate these combinations and then try to beat it later with more sophisticated approaches.
1) BoW  +  Logistic Regression
2) TF-IDF  +  Logistic Regression
3) multilingual pre-trained embeddings + Logistic Regression

Notes: MLP and Random Forest take way too long on such sparse and high dimensional embeddings as BoW and TF-IDF... Skipped for now

In [9]:
# import sys
# import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"])

In [10]:
import os
os.chdir('/home/taekim/Garnella')

In [11]:
%load_ext autoreload
%autoreload 2


In [12]:
import os
os.environ["PATH"] = "/cluster/courses/cil/envs/envs/text-classification/bin:" + os.environ.get("PATH", "")
import sys
print(sys.executable)

/cluster/courses/cil/envs/envs/text-classification/bin/python


In [13]:
import torch
print("PyTorch CUDA version:", torch.version.cuda)
print("GPU visible:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

PyTorch CUDA version: 12.8
GPU visible: True
GPU name: NVIDIA GeForce RTX 5060 Ti


In [14]:
from embeddings import *
from models import *
from train_loop_caching import train_loop

# Cell 1: Embedding comparison (same model, different embeddings)
combinations_embeddings = [
    #(get_tfidf_embeddings, get_logistic_regression),
   # (get_multilingual_embeddings, get_logistic_regression),
    #(get_gemma_embeddings, get_logistic_regression),
    #(get_qwen_embeddings, get_logistic_regression),
]
#results_emb = train_loop(train_df, val_df, combinations_embeddings)
#pd.DataFrame(results_emb).sort_values("validation_score", ascending=False)

/home/taekim/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# Cell 2: Model comparison (best embeddings, different classifiers)
combinations_models = [
    # (get_gemma_embeddings, get_logistic_regression),
    # (get_gemma_embeddings, get_linear_svm),
    # (get_gemma_embeddings, get_knn),
    # (get_gemma_embeddings, get_mlp),
    (get_gemma_embeddings, get_xgboost),
]
results_models = train_loop(train_df, val_df, combinations_models)
pd.DataFrame(results_models).sort_values("validation_score", ascending=False)

Loading saved embeddings from ./embedding_cache
Done with embeddings: get_gemma_embeddings


/home/taekim/.local/lib/python3.14/site-packages/xgboost/core.py:751: UserWarning: [16:08:14] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Combination: get_gemma_embeddings + XGBClassifier
Training Score: 0.9349, MAE: 0.2603, Accuracy: 0.7832
Validation Score: 0.8904, MAE: 0.4383, Accuracy: 0.6224


,vectorizer,model,training_score,training_mae,training_accuracy,validation_score,validation_mae,validation_accuracy
0,get_gemma_embeddings,XGBClassifier,0.934922,0.260313,0.783197,0.890437,0.438254,0.622381
